In [2]:
import pandas as pd
import json

df_train =pd.read_csv("train.csv")
df_sample = pd.read_csv("sample_submission.csv")

with open('music_app.json', 'r') as file:
    json_data = json.load(file)


In [3]:
null_count = df_train.isna().sum()
null_percentage = df_train.isna().mean() * 100

missing_df = pd.concat([null_count, null_percentage], axis = 1)
missing_df.columns = ["Missing count", "Missing%"]

missing_df[missing_df["Missing count"] > 0]


#null_count[null_count>0]

,Missing count,Missing%
music_app_user_id,1695,5.650000
stated_favourite_genre,4625,15.416667
typical_ticket_spend,8240,27.466667
usual_booking_lead_days,8240,27.466667
event_page_visits_7d,863,2.876667
travel_search_city,22496,74.986667
travel_search_arrival,22496,74.986667


In [4]:
df_engineered1= df_train.copy()

#put the date from object to date format

df_engineered1["sale_opened_at"]= pd.to_datetime(df_engineered1["sale_opened_at"])
df_engineered1["concert_at"] = pd.to_datetime(df_engineered1["concert_at"])
df_engineered1["prediction_at"]=pd.to_datetime(df_engineered1["prediction_at"])
df_engineered1["travel_search_arrival"]=pd.to_datetime(df_engineered1["travel_search_arrival"],errors="coerce")

# add a column to specify that the 0 for typical_ticket_spend correspond to missing value
df_engineered1["spend_missing"] = df_engineered1["typical_ticket_spend"].isna().astype(int)

# Replace the Nan of typical ticket spend to 0
df_engineered1[["typical_ticket_spend"]]= df_train[["typical_ticket_spend"]].fillna(0)


# List of columns to modify Nan for Unknown
colonnes_changetounknown = ["stated_favourite_genre", "travel_search_city"]

# Apply change to this column
df_engineered1[colonnes_changetounknown] = df_engineered1[colonnes_changetounknown].fillna("Unknown")

#add a column to specify that the Median for even_page Visit correspond to missing value
df_engineered1["Event_pages_missing"] = df_engineered1["event_page_visits_7d"].isna().astype(int)

#Modify Event_page_visits_7d NaN with the Median
df_engineered1["event_page_visits_7d"] = df_engineered1["event_page_visits_7d"].fillna(df_engineered1["event_page_visits_7d"].median())

#Add a numerical 0-1 column for Has Music app
df_engineered1["Has_music_app"] = df_engineered1["music_app_user_id"].notna().astype(int)

# Create an indicator for unknown dates for travel_search

df_engineered1["travel_search_date_unknown"] = df_engineered1["travel_search_arrival"].isna().astype(int)

#create a new data set for the model
X= df_engineered1.drop(columns="music_app_user_id")

# Drop redondant information coming from artist_id, artist_name, concert_id

X = X.drop(columns =["artist_id", "concert_id"])

#keep the user_id
groups = X["user_id"]

# Drop the identifiers user_id, row_id but the user information are still here

X = X.drop(columns=["row_id", "user_id"])


# replace the missing values for Usual_booking_lead_days by the median

X["usual_booking_lead_days"]= X["usual_booking_lead_days"].fillna(X["usual_booking_lead_days"].median())

# for the first model remove the dates

date_columns = ["concert_at", "sale_opened_at", "prediction_at", "travel_search_arrival"]

X = X.drop(columns =date_columns)


# One hot encode the categorical data to have numerical values 0 or 1

X_encoded = pd.get_dummies(X, columns=["education", "home_city", "stated_favourite_genre", "artist_name", "venue_city", "travel_search_city"], dtype=int)


#separate the genre column into different columns for concert genres numerical 1 or 0

concert_genres = X_encoded["genres"].str.get_dummies(sep="|")

concert_genres = concert_genres.add_prefix("concert_genre_")

X_encoded = pd.concat([X_encoded, concert_genres], axis=1)


#drop the original concert genres

X_encoded = X_encoded.drop(columns= "genres")

#Separate the features to predict value

y= X_encoded["buy"]

X_encoded = X_encoded.drop(columns="buy")




null_count = X.isna().sum()
null_count[null_count>0]


Series([], dtype: int64)

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# groupe shuffle split allow to keep the split by use, because each user have 5 rows

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, valid_idx = next(
    splitter.split(X_encoded, y, groups=groups)
)

X_train = X_encoded.iloc[train_idx]
X_valid = X_encoded.iloc[valid_idx]

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#scale the data
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)


In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

from sklearn.metrics import accuracy_score, roc_auc_score

y_pred = model.predict(X_valid_scaled)

y_proba = model.predict_proba(X_valid_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_valid, y_pred))
print("ROC-AUC:", roc_auc_score(y_valid, y_proba))

Accuracy: 0.674
ROC-AUC: 0.6659726734313971
